In [1]:
# Import your existing types and functions
from typing import Optional, Union
from dataclasses import dataclass
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

import pandas as pd
import torch
from torch import Tensor, einsum
from jaxtyping import Float, Int, Bool

from js_embedding_vis import write_inlined_config

from muutils.collect_warnings import CollateWarnings
from muutils.dbg import dbg, dbg_auto, dbg_tensor
import matplotlib.pyplot as plt


from spd.analysis.embed_vis import AnalysisConfig, coactivation_analysis, plot_embedding_label_grid
from spd.analysis.grouping import (
    CoactivationResults,
    CoactivationResultsGroup,
    get_coactivations,
    compute_merge_costs,
)
from spd.data_utils import SparseFeatureDataset
from spd.experiments.resid_mlp.resid_mlp_dataset import ResidualMLPDataset
from spd.utils import get_device_torch
from spd.analysis.merge_matrix import GroupMerge, BatchedGroupMerge

SyntaxError: expected ':' (grouping.py, line 460)

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
DEVICE = get_device_torch()
torch.set_grad_enabled(False)
print(f"Using device: {DEVICE = }")

In [ ]:
COACTIVATIONS = get_coactivations(
    model_path=Path("../data/mlp-decomp/model_30000.pth"),
    dataset_cls=ResidualMLPDataset,
    dataset_kwargs=dict(
        calc_labels=False,  # Our labels will be the output of the target model
        label_type=None,
        act_fn_name=None,
        label_fn_seed=None,
        label_coeffs=None,
        synced_inputs=None,
    ),
	dataloader_kwargs=dict(
		batch_size=100,
    ),
    coactivations_kwargs=dict(
        module_groups=[["layers.0.mlp_in", "layers.0.mlp_out"]],
		n_samples=100,
    ),
    device=DEVICE,
)["group_0"]



dbg_auto(COACTIVATIONS);

the above actually makes sense. the last 100 are the superimposed components which we expect to merge into one super-component.	

In [ ]:
# gm = GroupMerge.random(
#     n_components=200,
#     k_groups=50,
#     ensure_groups_nonempty=True,
# )
gm = GroupMerge.identity(n_components=200)

gm.plot(figsize=(10, 2))
gm_downstream: BatchedGroupMerge = gm.all_downstream_merged()
dbg_tensor(gm_downstream.group_idxs);
dbg("d")
gmd_m = gm_downstream.group_idxs
dbg("e")
dbg_tensor(gmd_m);
gm_downstream[0].plot(figsize=(10, 2))


In [ ]:
dbg_tensor(COACTIVATIONS['active_mask'])
merge_costs = compute_merge_costs(
    activation_mask=COACTIVATIONS['active_mask'],
    bgm=gm_downstream,
    alpha=1.0,
)
dbg_tensor(merge_costs);

In [ ]:
n_downstream: int = gm.k_groups
merge_costs_grid = torch.full((n_downstream, n_downstream), torch.nan)
for g in range(gm_downstream.batch_size):
    mp = gm_downstream.meta[g]['merge_pair']
    merge_costs_grid[mp] = merge_costs[g].cpu()
    merge_costs_grid[mp[1], mp[0]] = merge_costs[g].cpu()

plt.matshow(merge_costs_grid)
plt.colorbar(label='Merge Cost')

In [ ]:
def greedy_merge(
	activation_mask: Bool[Tensor, "n_samples n_components"],
	target_k_groups: int,
    alpha: float = 1.0,
	initial_guess: GroupMerge|None = None,
):
	n_samples: int; n_components: int
	n_samples, n_components = activation_mask.shape

	current_merge: GroupMerge
	if initial_guess is None:
		current_merge = GroupMerge.identity(n_components)
	else:
		current_merge = initial_guess


	dbg_tensor(current_merge.to_matrix())

	while current_merge.k_groups > target_k_groups:
		# Compute merge costs for all pairs of groups
		adm: BatchedGroupMerge = current_merge.all_downstream_merged()
		dbg_tensor(adm.to_matrix())
		merge_costs: Tensor = compute_merge_costs(
			activation_mask=activation_mask,
			bgm=adm,
			alpha=alpha,
		)

		# Find the pair with the lowest merge cost
		min_cost, min_pair = torch.min(merge_costs, dim=0)
		dbg(min_cost)
		dbg(min_pair)

		# Merge the pair with the lowest cost
		current_merge = current_merge.merge_groups(min_pair)
		dbg(f"Merging {min_pair} with cost {min_cost.item()}")
		dbg_tensor(current_merge.to_matrix())



greedy_merge(COACTIVATIONS['active_mask'], 10)

In [ ]:


def generate_all_merge_matrices(n_components: int, k: int, device=None) -> Bool[Tensor, "n_matrices k n_components"]:
    """Generate all possible merge matrices as single tensor"""
    if device is None:
        device = torch.device('cpu')
    
    # Total possibilities: k^n_components
    n_total = k ** n_components
    
    # Generate all base-k assignments
    assignments = torch.zeros(n_total, n_components, dtype=torch.long, device=device)
    
    for i in range(n_total):
        temp = i
        for j in range(n_components):
            assignments[i, j] = temp % k
            temp //= k
    
    # Convert to one-hot merge matrices
    merge_matrices = torch.zeros(n_total, k, n_components, dtype=torch.bool, device=device)
    batch_indices = torch.arange(n_total, device=device).unsqueeze(1)  # [n_total, 1]
    component_indices = torch.arange(n_components, device=device).unsqueeze(0)  # [1, n_components]
    
    merge_matrices[batch_indices, assignments, component_indices] = True
    
    # Filter out matrices with empty groups
    group_counts = merge_matrices.sum(dim=2)  # [n_total, k]
    valid_mask = (group_counts > 0).all(dim=1)  # [n_total]
    
    return merge_matrices[valid_mask]


def find_all_merge_costs(
    co_occurrence_matrix: Float[Tensor, "n_components n_components"],
    marginal_counts: Float[Tensor, "n_components"],
    k: int,
    alpha: float = 1.0,
) -> Float[Tensor, "n_matrices"]:
    """Return costs for all possible merge matrices"""
    all_matrices = generate_all_merge_matrices(marginal_counts.shape[0], k, marginal_counts.device)
    return compute_merge_costs(co_occurrence_matrix, marginal_counts, all_matrices, alpha)


def greedy_merge_search(
    co_occurrence_matrix: Float[Tensor, "n_components n_components"],
    marginal_counts: Float[Tensor, "n_components"],
    k: int,
    alpha: float = 1.0,
    temperature: float = 0.0,
    seed: Optional[int] = None,
) -> tuple[Bool[Tensor, "k n_components"], float]:
    """Greedy search with optional temperature sampling"""
    if seed is not None:
        torch.manual_seed(seed)
    
    n_components = marginal_counts.shape[0]
    device = marginal_counts.device
    
    # Start with identity
    merge_matrix = torch.eye(n_components, dtype=torch.bool, device=device)
    current_k = n_components
    
    while current_k > k:
        # Generate all possible merge candidates
        candidates = []
        
        for i in range(current_k):
            for j in range(i + 1, current_k):
                # Create candidate by merging groups i and j
                candidate = merge_matrix.clone()
                candidate[i] = candidate[i] | candidate[j]
                
                # Remove group j by shifting
                if j < current_k - 1:
                    candidate[j:current_k-1] = candidate[j+1:current_k]
                candidate = candidate[:current_k-1]
                
                candidates.append(candidate)
        
        # Compute costs for all candidates
        if len(candidates) > 0:
            candidate_stack = torch.stack(candidates)  # [n_candidates, current_k-1, n_components]
            
            # Compute current cost
            current_cost = compute_merge_costs(co_occurrence_matrix, marginal_counts, merge_matrix, alpha)
            
            # Compute candidate costs
            candidate_costs = compute_merge_costs(co_occurrence_matrix, marginal_counts, candidate_stack, alpha)
            cost_deltas = candidate_costs - current_cost
            
            # Select based on temperature
            if temperature == 0.0:
                best_idx = cost_deltas.argmin().item()
            else:
                probs = torch.softmax(-cost_deltas / temperature, dim=0)
                best_idx = torch.multinomial(probs, 1).item()
            
            merge_matrix = candidates[best_idx]
            current_k -= 1
        else:
            break
    
    final_cost = compute_merge_costs(co_occurrence_matrix, marginal_counts, merge_matrix, alpha)
    return merge_matrix, final_cost